# 3 — Merge, Enrichment & Cleaning

**Stage:** `raw_scraped.csv` + `raw_api.csv` → `data/clean/clean.csv`
**Requires:** `HARDCOVER_API_TOKEN` in `notebooks/.env`

## Purpose
Turn two partially-overlapping, partially-null raw sources into a single catalogue.

**Strategy — fill before you drop.** Every null is attacked in a fixed order before any row is
discarded:

```text
GoodReads scrape ─┐
                  ├─▶ cross-fill Google nulls from GoodReads (fuzzy title+author match)
Google Books API ─┘
                  ├─▶ concatenate
                  ├─▶ Open Library search.json   (fills rating, pages, year, genres, cover)
                  ├─▶ Hardcover GraphQL          (fills the same + synopsis)
                  └─▶ column-by-column cleaning, dedup, save
```

Each enrichment pass writes a boolean provenance flag (`gr_matched`, `ol_matched`,
`hc_matched`) recording whether that source touched the row.

**Conventions:** 🎯 goal · 🔍 observation · ✅ decision · ⚠️ limitation · 🤖 AI-assisted text, reviewed by the author

## 1. Setup & load the two raw sources

🎯 Load `raw_scraped` (GoodReads) and `raw_api` (Google Books) via the config-driven `read_file()` helper — no hard-coded paths. `summarise_dataframe()` prints shape, dtypes and null counts so the starting point is documented, not assumed.

In [2]:
# import function
%load_ext autoreload
%autoreload 2
from functions import read_file, out_csv, summarise_dataframe
import pandas as pd

yaml_path= "../config.yaml"

In [ ]:
#importing data file:
inp_data_section='raw_data'
file_name1= 'raw_scraped'    
file_name2= 'raw_api'    


gr_df= read_file(yaml_path, inp_data_section, file_name1) #good reads books
ggl_df = read_file(yaml_path, inp_data_section, file_name2) #google books
print('========================')
summarise_dataframe(gr_df)
print('========================')
summarise_dataframe(ggl_df)

## 2. Enrichment 1 — cross-fill Google nulls from GoodReads

🎯 The two sources overlap. Where Google Books has a null that GoodReads already holds, copy it across rather than paying for an API call.

**Matching logic — why fuzzy, and why two cutoffs:**

- `norm()` strips case, punctuation, bracketed subtitles and leading articles, so `"The Hobbit (Middle-Earth #1)"` and `"hobbit"` collide.
- With an author present: title similarity ≥ **0.90** *and* author similarity ≥ **0.85**. Two weak signals confirming each other.
- With no author: title alone must clear **0.97**. A single signal must be near-exact, otherwise sequels and reissues match each other.

✅ **Match-once, fill-many:** one match populates every fillable column at once. Only nulls are written — an existing value is never overwritten.

In [ ]:
import re
from difflib import SequenceMatcher, get_close_matches

TITLE_CUTOFF        = 0.90
TITLE_CUTOFF_STRICT = 0.97
AUTHOR_CUTOFF       = 0.85

FILL_COLS = ['author', 'rating', 'synopsis', 'pages', 'pub_date', 'genres', 'img_url']

def norm(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r'\([^)]*\)', '', s)
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    for art in ('the ', 'a ', 'an '):
        if s.startswith(art):
            s = s[len(art):]
    return s

def sim(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b).ratio()

gr_titles_norm  = gr_df['book_name'].map(norm).tolist()
gr_authors_norm = gr_df['author'].map(norm).tolist()

def find_gr_match(q_title, q_author):
    if not q_title:
        return None
    if q_author:
        for cand in get_close_matches(q_title, gr_titles_norm, n=5, cutoff=TITLE_CUTOFF):
            gr_idx = gr_titles_norm.index(cand)
            if sim(q_author, gr_authors_norm[gr_idx]) >= AUTHOR_CUTOFF:
                return gr_idx
        return None
    else:
        cand = get_close_matches(q_title, gr_titles_norm, n=1, cutoff=TITLE_CUTOFF_STRICT)
        return gr_titles_norm.index(cand[0]) if cand else None

ggl_df['gr_matched'] = False

nulls_before = {c: int(ggl_df[c].isna().sum()) for c in FILL_COLS}
fills        = {c: 0 for c in FILL_COLS}
matched_rows = 0

mask_any_null = ggl_df[FILL_COLS].isna().any(axis=1)
for i in ggl_df.index[mask_any_null]:
    gr_idx = find_gr_match(norm(ggl_df.at[i, 'book_name']),
                           norm(ggl_df.at[i, 'author']))
    if gr_idx is None:
        continue
    matched_rows += 1
    ggl_df.at[i, 'gr_matched'] = True
    gr_row = gr_df.iloc[gr_idx]
    for c in FILL_COLS:
        if pd.isna(ggl_df.at[i, c]) and pd.notna(gr_row[c]):
            ggl_df.at[i, c] = gr_row[c]
            fills[c] += 1

print("=== GR enrichment report ===")
print(f"Rows with >=1 fillable null : {int(mask_any_null.sum())}")
print(f"Rows matched to GoodReads   : {matched_rows}\n")
print(f"{'column':<10}{'nulls_before':>13}{'filled':>8}{'nulls_after':>13}")
for c in FILL_COLS:
    print(f"{c:<10}{nulls_before[c]:>13}{fills[c]:>8}{int(ggl_df[c].isna().sum()):>13}")

## 3. Concatenate the two sources

🎯 Stack GoodReads and (now partially filled) Google Books into one frame. `data_source` records which pipeline each row came from — it survives to the app.

In [ ]:
#Merging both sources
gr_df['data_source']='GoodReads'
concat_df = pd.concat([gr_df, ggl_df], ignore_index=True)
summarise_dataframe(concat_df)

## 4. Enrichment 2 — Open Library

🎯 Google Books returned still many null ratings. For every row still missing a fillable field, query Open Library on title + author.

**Why this API:** no key required, free, and it exposes `ratings_average`, `number_of_pages_median`, `first_publish_year`, `subject` and a cover ID in one call.

⚠️ **Constraints accepted:**
- `synopsis` is **not** available from `search.json` and is deliberately excluded   from `OL_FIELDS` — querying for it would return nothing.
- `subject` is extremely noisy; capped at the first 15 tags. Notebook 4 does the   real genre cleaning.
- Rows touched are flagged `ol_matched = True`.

In [ ]:
import time
import requests

OL_URL  = "https://openlibrary.org/search.json"
HEADERS = {"User-Agent": "IronhackW9-BookRec/1.0 (student project)"}
SLEEP   = 1.1

# synopsis excluded: not available via search.json
OL_FIELDS = ['author', 'rating', 'pages', 'pub_date', 'genres', 'img_url']

def ol_lookup(title: str, author: str) -> dict:
    """One OL query -> dict of available fields (only keys OL returned)."""
    params = {
        "title":  str(title),
        "fields": "ratings_average,number_of_pages_median,first_publish_year,"
                  "author_name,subject,cover_i",
        "limit":  1,
    }
    if pd.notna(author):
        params["author"] = str(author)
    out = {}
    try:
        r = requests.get(OL_URL, params=params, headers=HEADERS, timeout=15)
        r.raise_for_status()
        docs = r.json().get("docs", [])
        if not docs:
            return out
        d = docs[0]
        if d.get("ratings_average") is not None:
            out['rating'] = round(float(d["ratings_average"]), 2)
        if d.get("number_of_pages_median") is not None:
            out['pages'] = float(d["number_of_pages_median"])
        if d.get("first_publish_year") is not None:
            out['pub_date'] = str(d["first_publish_year"])      # year only
        if d.get("author_name"):
            out['author'] = d["author_name"][0]
        if d.get("subject"):
            out['genres'] = "|".join(d["subject"][:15])         # noisy, cap it
        if d.get("cover_i") is not None:
            out['img_url'] = f"https://covers.openlibrary.org/b/id/{d['cover_i']}-L.jpg"
    except Exception as e:
        print(f"  ⚠️ {str(title)[:40]!r}: {e}")
    return out

concat_df['ol_matched'] = False

mask_any_null = concat_df[OL_FIELDS].isna().any(axis=1)
query_idx = concat_df.index[mask_any_null]

nulls_before = {c: int(concat_df[c].isna().sum()) for c in OL_FIELDS}
fills        = {c: 0 for c in OL_FIELDS}
matched_rows = 0

print(f"Querying Open Library for {len(query_idx)} rows "
      f"(~{len(query_idx)*SLEEP/60:.1f} min)...\n")

for n, i in enumerate(query_idx, 1):
    data = ol_lookup(concat_df.at[i, "book_name"], concat_df.at[i, "author"])
    if data:
        got = False
        for c in OL_FIELDS:
            if c in data and pd.isna(concat_df.at[i, c]):
                concat_df.at[i, c] = data[c]
                fills[c] += 1
                got = True
        if got:
            matched_rows += 1
            concat_df.at[i, 'ol_matched'] = True
    if n % 25 == 0:
        print(f"  {n}/{len(query_idx)} checked, {matched_rows} rows filled")
    time.sleep(SLEEP)

print("\n=== Open Library enrichment report ===")
print(f"Rows queried      : {len(query_idx)}")
print(f"Rows filled (>=1) : {matched_rows}\n")
print(f"{'column':<10}{'nulls_before':>13}{'filled':>8}{'nulls_after':>13}")
for c in OL_FIELDS:
    print(f"{c:<10}{nulls_before[c]:>13}{fills[c]:>8}{int(concat_df[c].isna().sum()):>13}")

## 5. Checkpoint — save the concatenated + enriched catalogue

🎯 Persist to `raw_concatenated` so the two slow enrichment passes above never have to be re-run.

In [ ]:
print('========================')
concat_df = concat_df.drop(columns=['gr_matched','rating_source'])

In [ ]:
summarise_dataframe(concat_df)

In [ ]:
out_csv(
    df                  = concat_df,
    yaml_path           = yaml_path,
    output_section_yaml = 'raw_data',
    file_name           = 'raw_concatenated'
)

print(f'Saved {len(concat_df)} records.')

In [4]:
concat_df= read_file(yaml_path, inp_data_section='raw_data', file_name='raw_concatenated') #importing 
summarise_dataframe(concat_df)

Shape: (1298, 11)

--- Null counts ---
author       32
rating      332
synopsis    174
pages         7
pub_date     11
genres       50
img_url      42
dtype: int64

--- Dtypes ---
book_name          str
book_url           str
source_list        str
author             str
rating         float64
synopsis           str
pages          float64
pub_date           str
genres             str
img_url            str
data_source        str
dtype: object


## 6. Enrichment 3 — Hardcover

🎯 Third and final null-fill pass. Hardcover is the only one of the three back-fill sources that also returns a **synopsis**, which notebook 5 needs for mood scoring.

**Design notes:**
- `STEP 0` prints the raw response document *before* the loop runs — the schema is   verified, not assumed.
- ⚠️ **Rating gate:** Hardcover returns `rating = 0` for books with zero reviews.   Ingesting that as a real score would poison the rating column with fake zeros, so   `rating` is nulled whenever `ratings_count == 0`.
- Title similarity must clear **0.90**; no author check is applied here, so this pass   is looser than section 2.
- Rows touched are flagged `hc_matched = True`.

In [9]:
# ============================================================
# Hardcover API enrichment for concat_df
# Fills: author, rating, synopsis, pages, pub_date, genres, img_url
# ============================================================

import os, json, time, requests
import pandas as pd
from difflib import SequenceMatcher
from dotenv import load_dotenv

load_dotenv()  # reads notebooks/.env
HC_TOKEN = os.environ["HARDCOVER_API_TOKEN"]
HC_URL   = "https://api.hardcover.app/v1/graphql"
HEADERS  = {"authorization": HC_TOKEN, "content-type": "application/json"}

SEARCH_Q = """
query Search($q: String!) {
  search(query: $q, query_type: "Book", per_page: 5, page: 1) { results }
}
"""

def hc_search(title):
    r = requests.post(HC_URL, headers=HEADERS,
                       json={"query": SEARCH_Q, "variables": {"q": title}}, timeout=30)
    r.raise_for_status()
    return r.json()

# STEP 0 schema check. 
# below match the printed doc before running next step.
# ------------------------------------------------------------
sample = hc_search("The Hobbit")
hits = sample["data"]["search"]["results"]["hits"]
print(json.dumps(hits[0]["document"], indent=2)[:1500])

{
  "activities_count": 2,
  "alternative_titles": [
    "The Hobbit, Part Two"
  ],
  "author_names": [
    "J.R.R. Tolkien",
    "David Wyatt"
  ],
  "book_category_id": 1,
  "compilation": false,
  "content_warnings": [],
  "contribution_types": [
    "Author",
    "Illustrator"
  ],
  "contributions": [
    {
      "author": {
        "id": 132049,
        "image": {
          "color": "#5a5a5a",
          "color_name": "Gray",
          "height": 266,
          "id": 33205,
          "url": "https://assets.hardcover.app/authors/132049/6155606-L.jpg",
          "width": 187
        },
        "name": "J.R.R. Tolkien",
        "slug": "j-r-r-tolkien"
      },
      "contribution": null
    },
    {
      "author": {
        "id": 251116,
        "image": {},
        "name": "David Wyatt",
        "slug": "david-wyatt"
      },
      "contribution": "Illustrator"
    }
  ],
  "cover_color": "Brown",
  "description": "The Hobbit is a tale of high adventure, undertaken by a company of 

In [10]:
# ----
# STEP 1 — enrichment loop
# ------------------------------------------------------------
HC_FIELDS    = ['author', 'rating', 'synopsis', 'pages', 'pub_date', 'genres', 'img_url']
TITLE_CUTOFF = 0.90
SLEEP        = 1.1   # 60 req/min hard limit

def _norm(s):
    return str(s).strip().lower() if s is not None else ""

def _pick(doc):
    authors = doc.get("author_names") or []
    image   = doc.get("image") or {}
    genres  = doc.get("genres") or []
    
    # ⚠️ GATE rating on ratings_count > 0, else it's a fake zero
    rating = doc.get("rating")
    if rating is not None and doc.get("ratings_count", 0) == 0:
        rating = None   # no real data on Hardcover
    
    return {
        "author":   authors[0] if authors else None,
        "rating":   rating,                           # now gated
        "synopsis": doc.get("description"),
        "pages":    doc.get("pages"),
        "pub_date": doc.get("release_year"),
        "genres":   "|".join(genres) if genres else None,
        "img_url":  image.get("url"),
    }

def hc_best_match(title):
    try:
        hits = hc_search(title)["data"]["search"]["results"]["hits"]
    except Exception as e:
        print(f"  ! search failed for {title!r}: {e}")
        return None
    tnorm, best, best_score = _norm(title), None, 0.0
    for h in hits:
        doc = h.get("document", {})
        score = SequenceMatcher(None, tnorm, _norm(doc.get("title"))).ratio()
        if score > best_score:
            best_score, best = score, doc
    return _pick(best) if best_score >= TITLE_CUTOFF else None

mask_any_null = concat_df[HC_FIELDS].isna().any(axis=1)
n_target = int(mask_any_null.sum())
print(f"Rows to query: {n_target}  (~{n_target * SLEEP / 60:.1f} min)")

concat_df["hc_matched"] = False

for i, idx in enumerate(concat_df.index[mask_any_null]):
    title = concat_df.at[idx, "book_name"]
    if not str(title).strip():
        continue
    fields = hc_best_match(title)
    time.sleep(SLEEP)
    if fields is None:
        continue
    concat_df.at[idx, "hc_matched"] = True
    for col in HC_FIELDS:
        if pd.isna(concat_df.at[idx, col]) and fields.get(col) not in (None, ""):
            concat_df.at[idx, col] = fields[col]
    if (i + 1) % 50 == 0:
        print(f"  ...{i+1}/{n_target} processed")

print("HC matched:", int(concat_df["hc_matched"].sum()))

# --------------------
# STEP 2 — re-check nulls after
# ------------------------------------------------------------
print("\n--- Null counts after Hardcover ---")
print(concat_df[HC_FIELDS].isna().sum())
print(f"\nCoverage: {concat_df['hc_matched'].mean():.1%} of queried rows matched")

Rows to query: 383  (~7.0 min)
  ...50/383 processed
  ...200/383 processed
  ...250/383 processed
  ! search failed for 'Letters of Blood and Other Works in English': 'NoneType' object is not subscriptable
HC matched: 210

--- Null counts after Hardcover ---
author       32
rating      242
synopsis    132
pages         6
pub_date     11
genres       44
img_url      30
dtype: int64

Coverage: 16.2% of queried rows matched


## 7. Checkpoint — save `raw_combined`

🎯 All three enrichment passes are complete. Save before cleaning begins, so cleaning can be re-run from a fixed input.

In [11]:
concat_df.columns

Index(['book_name', 'book_url', 'source_list', 'author', 'rating', 'synopsis',
       'pages', 'pub_date', 'genres', 'img_url', 'data_source', 'hc_matched'],
      dtype='str')

In [12]:
concat_df = concat_df.drop(columns=['hc_matched'])

In [13]:
out_csv(
    df                  = concat_df,
    yaml_path           = yaml_path,
    output_section_yaml = 'raw_data',
    file_name           = 'raw_combined'
)

print(f'Saved {len(concat_df)} records.')

File saved to: ../data/raw/all_books.csv
Saved 1298 records.


## 8. Cleaning — genres

🎯 From here on the notebook goes column by column. Reload from `raw_combined` so cleaning is independent of the enrichment cells above.

🔍 Books still missing `genres` all come from a small number of `source_list` values.

In [63]:
combined_df= read_file(yaml_path, inp_data_section='raw_data', file_name='raw_combined') #importing 
books_df=combined_df.copy()
books_df[books_df.genres.isnull()].source_list.unique()

<ArrowStringArray>
['Best Dystopian and Post-Apocalyptic Fiction',
                             'Best Books Ever',
                       'Books You Should Read',
                                 'Young Adult',
                 'Books That Should Be Movies',
                             'featured Google']
Length: 6, dtype: str

✅ **Decision — fill genres from the list they were found on.** A book scraped from *'Best Dystopian and Post-Apocalyptic Fiction'* is, at minimum, dystopian fiction. Lists with no genre signal (*'Best Books Ever'*, *'Young Adult'*) are mapped to `'general'` rather than invented — the tag is honest about carrying no information.

In [64]:
#define your mapping dictionary (List Name -> Genre)
genre_mapping = {
    'Best Dystopian and Post-Apocalyptic Fiction': 'Fiction|Dystopian',
    'Best Books Ever': 'general',
    'Books You Should Read': 'general',
    'Young Adult': 'general',
    'Books That Should Be Movies': 'adventure|fiction',
    'featured Google': 'general',
}

#Fill missing values in 'genres' by mapping the 'source_list' column
books_df['genres'] = books_df['genres'].fillna(books_df['source_list'].map(genre_mapping))
books_df[books_df.genres.isnull()].source_list.unique()

<ArrowStringArray>
[]
Length: 0, dtype: str

## 9. Cleaning — author

🔍 Inspect the rows with no author first.

✅ **Decision:** drop them. Inspection shows they are journals, indexes and directories — not books. A recommender for books should not contain them.

In [65]:
books_df[books_df.author.isnull()]['book_name']

1190                                   The School Journal
1194    Bulletin of Bibliography and Magazine Subject-...
1206    Managing for Featured, Threatened, Endangered,...
1211                             The Sheet Music Exchange
1212                                       Film Year Book
1213                                  Paper Trade Journal
1214      United States Economist, and Dry Goods Reporter
1215                       Electrical Installation Record
1217      Florists Exchange and Horticultural Trade World
1219                      The Corset and Underwear Review
1220                              Exhibitors Daily Review
1227                                 Premier Rayon Review
1231    Minneapolis-Honeywell Regulator Company V. Fed...
1232    Records & Briefs New York State Appellate Divi...
1233                                      The Log Analyst
1234                                Music Trade Indicator
1238                            West's Federal Supplement
1242          

In [66]:
#droping rows with missing author as they are not books
books_df = books_df[books_df.author.notnull()]
books_df.shape

(1266, 11)

## 10. Cleaning — publication date

🎯 `pub_date` arrives in three shapes across the sources. Truncating to the first four characters normalises all of them to a year, which is the only granularity the app uses.

✅ A handful of remaining nulls are repaired manually from verified sources; a few non-book entries are dropped.

⚠️ **Fragility:** the manual repairs address rows by their **integer index** (`books_df.at[581, 'pub_date']`). Those indices are only valid for this exact run of this exact pipeline. Re-scrape the data and they point at different books.

In [67]:
books_df[books_df.pub_date.isnull()]['book_name']

581                                        The Cheat Code
843                   A Field Guide to the Addo Elephants
893                                           Tigerhjerte
915                            Skildpadder hele vejen ned
925                          Hvor solen skinner om natten
929                            Sådan opfinder man en pige
940                                               Worbook
954                                           The Pageant
1204    An Internet of Things Approach for Extracting ...
1207    Featured Reviews in "Mathematical Reviews" 199...
Name: book_name, dtype: str

In [68]:
books_df.pub_date.unique()

<ArrowStringArray>
['2008-09-14', '1813-01-28', '1960-07-11', '2003-06-21', '2005-09-01',
 '2005-10-05', '1945-08-17', '1954-01-01', '1956-01-01', '1890-07-01',
 ...
       '1960',       '1962',    '1996-02',       '1926',       '1928',
    '1988-08',       '1921',       '1923',       '1955',       '1922']
Length: 854, dtype: str

In [69]:
#isolate the year
books_df['pub_date'] = books_df['pub_date'].str[:4]

#Update the missing records using string years
valid_pub_dates = {
    581: '2016',
    843: '2002',
    893: '2018',
    915: '2017',
    925: '2018',
    929: '2015'
}

for idx, year in valid_pub_dates.items():
    if idx in books_df.index:
        books_df.at[idx, 'pub_date'] = year

#drop the entries that are not individual books
indices_to_drop = [940, 954, 1204, 1207]
books_df.drop(index=indices_to_drop, errors='ignore', inplace=True)

# Verify the result
print(books_df['pub_date'].unique())

<ArrowStringArray>
['2008', '1813', '1960', '2003', '2005', '1945', '1954', '1956', '1890',
 '1847',
 ...
 '1906', '1904', '1833', '1918', '1888', '1914', '1941', '1920', '1921',
 '1923']
Length: 152, dtype: str


In [70]:
books_df.isnull().sum()

book_name        0
book_url         0
source_list      0
author           0
rating         208
synopsis        98
pages            6
pub_date         0
genres           0
img_url         28
data_source      0
dtype: int64

## 11. Cleaning — cover image

✅ **Decision:** nulls are filled with a local placeholder image rather than dropped. A missing cover is a UI problem, not a data problem — the book is still recommendable.

In [71]:
# Define the relative path to missing cover image
coverimg_path = 'figures/missingcover.png'
# Fill the missing values in the img_url column
books_df['img_url'] = books_df['img_url'].fillna(coverimg_path)

In [72]:
books_df.isna().sum()   # sanity check nulls 

book_name        0
book_url         0
source_list      0
author           0
rating         208
synopsis        98
pages            6
pub_date         0
genres           0
img_url          0
data_source      0
dtype: int64

## 12. Cleaning — pages

🎯 `pages` is a user-facing filter in the app, so a null makes the book invisible to any page-range query.

✅ **Decision:** repair the few verifiable ones by hand, then drop the rest. Same index-fragility caveat as before.

In [73]:
books_df[books_df.pages.isnull()]

,book_name,book_url,source_list,author,rating,synopsis,pages,pub_date,genres,img_url,data_source
820,House of the Seven Gables [eBook - RBdigital],http://books.google.de/books?id=5YbGxgEACAAJ&d...,Books You Should Read,Nathaniel Hawthorne,NaN,"The House of the Seven Gables, a fixture in Ha...",NaN,1993,Domestic fiction,http://books.google.com/books/content?id=5YbGx...,google_books
893,Tigerhjerte,http://books.google.de/books?id=_wnztwEACAAJ&d...,Young Adult,Lise Villadsen,NaN,NaN,NaN,2018,general,figures/missingcover.png,google_books
915,Skildpadder hele vejen ned,http://books.google.de/books?id=hK-ZtAEACAAJ&d...,Young Adult,John Green,NaN,NaN,NaN,2017,general,figures/missingcover.png,google_books
925,Hvor solen skinner om natten,http://books.google.de/books?id=PaAotAEACAAJ&d...,Young Adult,Annie Bahnson,NaN,NaN,NaN,2018,general,figures/missingcover.png,google_books
929,Sådan opfinder man en pige,http://books.google.de/books?id=icvMsgEACAAJ&d...,Young Adult,Caitlin Moran,NaN,NaN,NaN,2015,general,figures/missingcover.png,google_books
952,Remote Angel,http://books.google.de/books?id=j9v9zgEACAAJ&d...,Young Adult,Yennie Fer,NaN,"""After a freak accident of an attempted assaul...",NaN,2022,Angels,figures/missingcover.png,google_books


In [74]:
#dictionary mapping indices to verified page numbers
verified_pages = {
    893: 288,
    915: 288,
    925: 264,
    929: 390
}

#update the 'pages' column safely
for idx, pages in verified_pages.items():
    if idx in books_df.index:
        books_df.at[idx, 'pages'] = pages

# Verify the changes
books_df[books_df.pages.isnull()]

,book_name,book_url,source_list,author,rating,synopsis,pages,pub_date,genres,img_url,data_source
820,House of the Seven Gables [eBook - RBdigital],http://books.google.de/books?id=5YbGxgEACAAJ&d...,Books You Should Read,Nathaniel Hawthorne,NaN,"The House of the Seven Gables, a fixture in Ha...",NaN,1993,Domestic fiction,http://books.google.com/books/content?id=5YbGx...,google_books
952,Remote Angel,http://books.google.de/books?id=j9v9zgEACAAJ&d...,Young Adult,Yennie Fer,NaN,"""After a freak accident of an attempted assaul...",NaN,2022,Angels,figures/missingcover.png,google_books


In [75]:
#droping rows with missing pages
books_df = books_df[books_df.pages.notnull()]
books_df.shape

(1260, 11)

## 13. Deduplication

🎯 The same book reaches the catalogue by several routes: multiple Listopia lists, and both sources at once. Duplicates appear under two different keys — identical `book_name`, and identical `synopsis` (re-issues, annotated editions, box sets).

In [76]:
books_df[books_df['book_name'].duplicated(keep=False)]

,book_name,book_url,source_list,author,rating,synopsis,pages,pub_date,genres,img_url,data_source
1,Pride and Prejudice,https://www.goodreads.com/book/show/1885.Pride...,Best Books Ever,Jane Austen,4.30,This comedy of manners about the games of Eros...,279.0,1813,Classics|Romance|Fiction|Historical Fiction|Hi...,https://m.media-amazon.com/images/S/compressed...,GoodReads
2,To Kill a Mockingbird,https://www.goodreads.com/book/show/2657.To_Ki...,Best Books Ever,Harper Lee,4.26,"""Shoot all the bluejays you want, if you can h...",323.0,1960,Classics|Fiction|Historical Fiction|School|Lit...,https://m.media-amazon.com/images/S/compressed...,GoodReads
3,Harry Potter and the Order of the Phoenix,https://www.goodreads.com/book/show/2.Harry_Po...,Best Books Ever,J.K. Rowling,4.50,"The fifth book in the beloved, bestselling Har...",576.0,2003,Fantasy|Fiction|Young Adult|Harry Potter|Magic...,https://m.media-amazon.com/images/S/compressed...,GoodReads
9,The Picture of Dorian Gray,https://www.goodreads.com/book/show/5297.The_P...,Best Books Ever,Oscar Wilde,4.13,"In this celebrated work, his only novel, Wilde...",272.0,1890,Classics|Fiction|Horror|Gothic|Fantasy|Literat...,https://m.media-amazon.com/images/S/compressed...,GoodReads
10,Wuthering Heights,https://www.goodreads.com/book/show/6185.Wuthe...,Best Books Ever,Emily Brontë,3.89,You can find the redesigned cover of this edit...,464.0,1847,Classics|Fiction|Romance|Gothic|Historical Fic...,https://m.media-amazon.com/images/S/compressed...,GoodReads
...,...,...,...,...,...,...,...,...,...,...,...
1169,Harry Potter and the Half-Blood Prince,http://books.google.de/books?id=OWIauAEACAAJ&d...,Best Epic Fantasy,J. K. Rowling,4.58,Sixth-year Hogwarts student Harry Potter gains...,652.0,2005,Juvenile Fiction,http://books.google.com/books/content?id=OWIau...,google_books
1171,Watership Down,http://books.google.de/books?id=X_OSuwsF4_UC&d...,Best Epic Fantasy,Richard Adams,5.00,Heroic rabbits trek from their poisoned warren...,582.0,1973,Fiction,http://books.google.com/books/content?id=X_OSu...,google_books
1173,"The Lion, the Witch and the Wardrobe",http://books.google.de/books?id=WWMrAAAAYAAJ&d...,Best Epic Fantasy,Clive Staples Lewis,4.13,"The best-selling rack edition of The Lion, the...",204.0,1970,Juvenile Fiction,http://books.google.com/books/content?id=WWMrA...,google_books
1175,The Eyes of the Dragon,http://books.google.de/books?id=KKcOAQAAMAAJ&d...,Best Epic Fantasy,Stephen King,3.94,A kingdom is in turmoil as the old king dies a...,344.0,1987,Fiction,http://books.google.com/books/content?id=KKcOA...,google_books


In [77]:
# Filter and display all copies of rows with duplicate synopses
duplicated_synopses = books_df[books_df['synopsis'].duplicated(keep=False)]

#sort by synopsis so the duplicates appear right next to each other
duplicated_synopses.sort_values(by='synopsis')

,book_name,book_url,source_list,author,rating,synopsis,pages,pub_date,genres,img_url,data_source
895,Developmental Oncology: Principles and Therapy...,http://books.google.de/books?id=dg7X0AEACAAJ&d...,Young Adult,Alex Kentsis,NaN,"""Hundreds of thousands of children develop leu...",0.0,2025,Medical,http://books.google.com/books/content?id=dg7X0...,google_books
919,Developmental Oncology,http://books.google.de/books?id=KFgP0QEACAAJ&d...,Young Adult,Alejandro Gutierrez,NaN,"""Hundreds of thousands of children develop leu...",0.0,2025,Medical,figures/missingcover.png,google_books
2,To Kill a Mockingbird,https://www.goodreads.com/book/show/2657.To_Ki...,Best Books Ever,Harper Lee,4.26,"""Shoot all the bluejays you want, if you can h...",323.0,1960,Classics|Fiction|Historical Fiction|School|Lit...,https://m.media-amazon.com/images/S/compressed...,GoodReads
85,To Kill a Mockingbird,https://www.goodreads.com/book/show/2654.To_Ki...,Best Books Ever,Harper Lee,4.26,"""Shoot all the bluejays you want, if you can h...",323.0,1960,Classics|Fiction|Historical Fiction|School|Lit...,https://m.media-amazon.com/images/S/compressed...,GoodReads
14,The Little Prince,https://www.goodreads.com/book/show/157993.The...,Best Books Ever,Antoine de Saint-Exupéry,4.34,A pilot forced to land in the Sahara meets a l...,96.0,1943,Classics|Fiction|Fantasy|Childrens|France|Phil...,https://m.media-amazon.com/images/S/compressed...,GoodReads
...,...,...,...,...,...,...,...,...,...,...,...
1289,Architectural Record,http://books.google.de/books?id=TihQAQAAIAAJ&d...,featured Google,Waverly Lowell,NaN,NaN,1016.0,1985,Architecture,http://books.google.com/books/content?id=TihQA...,google_books
1292,Domestic Engineering,http://books.google.de/books?id=bPLWQ_Cc8tcC&d...,featured Google,Frank W. Raynes,NaN,NaN,2162.0,1928,Domestic engineering,http://books.google.com/books/content?id=bPLWQ...,google_books
1293,Broadside,http://books.google.de/books?id=OkbaAAAAMAAJ&d...,featured Google,Theodore Kautzky,5.00,NaN,532.0,1967,"Ballads, English",http://books.google.com/books/content?id=OkbaA...,google_books
1294,Strategies,http://books.google.de/books?id=5WFLAAAAYAAJ&d...,featured Google,Martha Wells,4.62,NaN,248.0,2002,Physical education and training,http://books.google.com/books/content?id=5WFLA...,google_books


✅ **Decision — keep the most complete copy, not the first one.**

1. `pub_date` → true `datetime`.
2. `completeness_score` = count of non-null fields per row.
3. Sort by completeness (desc), then recency (desc).
4. `drop_duplicates` on `book_name`, then on `synopsis`, keeping the first — which,    after the sort, is the richest surviving record.
5. Drop the temporary scoring column.

This is why the sort must happen **before** the drop: `keep='first'` is only meaningful once 'first' has been defined as 'best'.

In [83]:
#directly convert the existing year string into a true datetime object
books_df['pub_date'] = pd.to_datetime(books_df['pub_date'], format='%Y', errors='coerce')

# count how many non-null values each row has across all columns
books_df['completeness_score'] = books_df.notna().sum(axis=1)

# sort by completeness (highest first), then by pub_date (newest year first)
books_df = books_df.sort_values(
    by=['completeness_score', 'pub_date'], 
    ascending=[False, False]
)

#drop duplicate book names (keeping the most complete/newest)
final_df = books_df.drop_duplicates(subset='book_name', keep='first')

# drop duplicate synopses from the already filtered final_df
final_df = final_df.drop_duplicates(subset='synopsis', keep='first')

#clean up by dropping the temporary scoring column
final_df = final_df.drop(columns=['completeness_score'])

# Verify 
print(final_df.dtypes)
print(final_df.shape)

book_name                 str
book_url                  str
source_list               str
author                    str
rating                float64
synopsis                  str
pages                 float64
pub_date       datetime64[us]
genres                    str
img_url                   str
data_source               str
dtype: object
(1089, 11)


## 14. Final manual repairs

🎯 A residual handful of nulls survive all three APIs. They are repaired by hand, and remaining non-books are removed.

In [87]:
final_df[final_df.synopsis.isnull()]

,book_name,book_url,source_list,author,rating,synopsis,pages,pub_date,genres,img_url,data_source
984,Malory Towers,http://books.google.de/books?id=9G8l0QEACAAJ&d...,Books That Should Be Movies,Enid Blyton,3.86,NaN,0.0,2016-01-01,Adventure,https://covers.openlibrary.org/b/id/14596326-L...,google_books


In [88]:
final_df[final_df.pub_date.isnull()]

,book_name,book_url,source_list,author,rating,synopsis,pages,pub_date,genres,img_url,data_source
783,Sense and Sensibility,http://books.google.de/books?id=PRzAuQEACAAJ&d...,Books You Should Read,Jane Austen,3.92,When two sisters appear to be deserted by the ...,352.0,NaT,England,http://books.google.com/books/content?id=PRzAu...,google_books
825,The House of Mirth,http://books.google.de/books?id=1MD3twEACAAJ&d...,Books You Should Read,Edith Wharton,4.12,"What Galsworthy did for Edwardian England, Wha...",348.0,NaT,New York (N.Y.),http://books.google.com/books/content?id=1MD3t...,google_books


In [89]:
#Fill the missing synopsis
if 984 in final_df.index:
    final_df.at[984, 'synopsis'] = (
        "Darrell Rivers begins her first term at Malory Towers, a castle-like clifftop "
        "boarding school in Cornwall. Eager to fit in, make friends, and succeed, she "
        "soon discovers that mastering her fierce temper is just as challenging as her schoolwork."
    )

#fill the missing publication dates (parsed as datetime)
if 783 in final_df.index:
    final_df.at[783, 'pub_date'] = pd.to_datetime('1811-10-30', errors='coerce')

if 825 in final_df.index:
    final_df.at[825, 'pub_date'] = pd.to_datetime('1905-10-14', errors='coerce')

# verify that no NaNs remain for these specific rows
print(final_df.loc[[984, 783, 825], ['book_name', 'synopsis', 'pub_date']])

                 book_name                                           synopsis  \
984          Malory Towers  Darrell Rivers begins her first term at Malory...   
783  Sense and Sensibility  When two sisters appear to be deserted by the ...   
825     The House of Mirth  What Galsworthy did for Edwardian England, Wha...   

      pub_date  
984 2016-01-01  
783 1811-10-30  
825 1905-10-14  


In [93]:
missing_rating_books = final_df[final_df['rating'].isnull()]['book_name'].unique().tolist()
print(missing_rating_books)

['Olken Ridge', 'Trauma, Pedagogy, and the College Mental Health Crisis', 'Developmental Oncology: Principles and Therapy of Cancers of Children and Young Adults', 'Si Se Lo Hubiera Dicho / If Only I Had Told Her', 'Soul Eater: The Perfect Edition 17', 'Social Media and Youth Mental Health', 'Make Me a Liar', 'Attachment-Based Family Therapy for Sexual and Gender Minority Young Adults and Their Non-Accepting Parents', 'Middlemarch: A Study of Provincial Life', 'The Legacy of the Jaded Hart', 'Hopeless in Hope', 'Emerging Adulthood', 'Little Lord Fauntleroy Illustrated Edition', 'The Reflections', 'Neuro-behavioral Manifestations of Prader-Willi Syndrome', 'Wuthering Heights Annotated', 'Typee Illustrated', 'Clinical Allergy and Asthma Management in Adolescents and Young Adults', 'The Innocence of Father Brown Illustrated', 'The Little Lady of the Big House Illustrated', 'The Hunchback of Notre Dame Annotated', 'The Delusionist', 'Robinson Crusoe Annotated', 'Anne of the Island Illustra

⚠️ **Rating repairs — read this before trusting the `rating` column.**

The ~125 ratings below were **looked up manually via an LLM-assisted web search**, not returned by any of the three APIs. That means:

- They are **not reproducible** — re-running this notebook does not regenerate them.
- They are **not verifiable against a source** — no URL or timestamp is recorded.
- They carry **no provenance flag**, so downstream code cannot tell them apart from   a real GoodReads or Open Library rating.

This is a documented trade-off, made to keep ~125 books in the catalogue rather than drop them. It is stated here, in the README and in the presentation — it is not hidden. A stricter alternative would be to drop these rows, or to add a `rating_source = 'manual'` flag and exclude them from any rating-based ranking.

In [94]:
import numpy as np

#dictionary mapping book names to their verified average ratings (float) based by Gemini search online.
ratings_map = {
    'Olken Ridge': 4.33,
    'Trauma, Pedagogy, and the College Mental Health Crisis': 3.67,
    'Developmental Oncology: Principles and Therapy of Cancers of Children and Young Adults': 4.00,
    'Si Se Lo Hubiera Dicho / If Only I Had Told Her': 3.73,
    'Soul Eater: The Perfect Edition 17': 4.49,
    'Social Media and Youth Mental Health': 4.25,
    'Make Me a Liar': 3.88,
    'Attachment-Based Family Therapy for Sexual and Gender Minority Young Adults and Their Non-Accepting Parents': 4.80,
    'Middlemarch: A Study of Provincial Life': 4.01,
    'The Legacy of the Jaded Hart': 4.15,
    'Hopeless in Hope': 3.72,
    'Emerging Adulthood': 3.96,
    'Little Lord Fauntleroy Illustrated Edition': 3.82,
    'The Reflections': 3.65,
    'Neuro-behavioral Manifestations of Prader-Willi Syndrome': 4.50,
    'Wuthering Heights Annotated': 3.89,
    'Typee Illustrated': 3.80,
    'Clinical Allergy and Asthma Management in Adolescents and Young Adults': 4.00,
    'The Innocence of Father Brown Illustrated': 3.86,
    'The Little Lady of the Big House Illustrated': 3.42,
    'The Hunchback of Notre Dame Annotated': 4.00,
    'The Delusionist': 3.62,
    'Robinson Crusoe Annotated': 3.69,
    'Anne of the Island Illustrated': 4.25,
    'Don Quixote Annotated': 4.12,
    'The Hunchback of Notre Dame (Annotated)': 4.00,
    'The Boy Who Sailed with Blake': 3.50,
    'Wuthering Heights Illustrated': 3.89,
    'The Valley of Fear Illustrated': 3.94,
    'The Three Musketeers Annotated': 4.08,
    'The Sorrows of Satan Illustrated': 4.07,
    'The Picture of Dorian Gray Illustrated': 4.12,
    'The Chimes Illustrated': 3.56,
    'Robinson Crusoe Illustrated': 3.69,
    'Pygmalion Illustrated': 3.95,
    'Martin Eden by Jack London': 4.18,
    'Moll Flanders Illustrated': 3.52,
    'As You Like It Annotated': 3.84,
    'Pride and Prejudice Annotated': 4.28,
    'Unforgettable You': 3.97,
    'The Me-generation in a Post-collectivist Space': 4.00,
    'The Power of Movement in Plants': 4.03,
    'Emerging Adults and Substance Use Disorder Treatment': 4.00,
    'Ruse and Romance': 4.11,
    'Sense and Sensibility - Illustrated Edition': 4.08,
    'Rhys Lewis, Minister of Bethel': 3.85,
    'Alice in Wonderland and Through the Looking Glass': 4.13,
    'Game Seven': 3.78,
    'The Readers’ Advisory Guide to Historical Fiction': 3.60,
    'Ancient Guardians': 4.12,
    'Me and Mr Jones': 3.65,
    'Hollowed Humusara': 4.50,
    "Harry Potter Box Set: the Complete Collection (Children's Paperback)": 4.74,
    'The Tutu': 3.86,
    'Whose History?': 3.67,
    'Avalanche of the Undead': 3.75,
    "100 Questions You'd Never Ask Your Parents": 3.40,
    'Young Adult Health': 4.00,
    'Treating Complex Trauma in Adolescents and Young Adults': 4.38,
    'Letters of Blood and Other Works in English': 4.41,
    'Invincible Iron Man - Stark Resilient': 3.82,
    'Tuning in': 3.80,
    'Sudden Death in the Young': 5.00,
    'Stroke in Children and Young Adults': 4.00,
    'Cerebral Ischemia in Young Adults': 4.00,
    'The Adolescent and Young Adult Self-harming Treatment Manual': 4.50,
    'Inventing Adulthoods': 3.50,
    'The Northern Devil': 3.89,
    'Chinese Theories of Fiction': 4.00,
    "Self's Deception": 3.70,
    'The Life and Adventures of Robinson Crusoe, of York, Mariner': 3.69,
    'Adventures in Extreme Grace Series': 4.50,
    'Klassische Texte zum Raum': 4.00,
    "There's No Toilet Paper ... on the Road Less Traveled": 3.62,
    'The Lion, the Witch and the Wardrobe Movie Tie-in Edition (adult)': 4.23,
    'Death of an Ancient King': 3.92,
    '西游记': 4.27,
    'J.J., Some Jottings': 4.00,
    'The Damned (La Bas)': 3.85,
    'South with Endurance': 4.44,
    'Harry Potter y la piedra filosofal': 4.48,
    'Moomin, Mymble and Little My': 4.45,
    'When Girls Feel Fat': 3.45,
    'Sociology': 3.81,
    'All Chalked Up': 3.82,
    'Through the Storm': 3.86,
    "John Brookes' Natural Landscapes": 4.20,
    'Parliament and the Media': 3.00,
    'Expanding Education & Literacy': 4.00,
    'Renaissance 2000': 3.50,
    'The Heart of a Dog': 4.09,
    'Homely Girl, a Life, and Other Stories': 3.54,
    "Professor Martens' Departure": 3.69,
    'Through the Looking Glass and what Alice Found There': 4.07,
    'Children Exploring Their World': 3.60,
    'Still Loved by the Sun': 4.00,
    'Underrunners': 3.65,
    'Hemingway': 3.91,
    'Little Follies': 4.07,
    'Boy Trouble': 3.52,
    'Expert Systems in Computer-aided Design': 4.00,
    'Casanova\'s "Icosameron", Or, The Story of Edward and Elizabeth': 3.45,
    'The Last Museum': 3.50,
    'Observing the Erotic Imagination': 4.20,
    'The Murder at Hazelmoor': 3.89,
    'Joanna': 3.71,
    'Thirteen at Dinner': 3.98,
    'MacIntosh Mountain': 3.90,
    'No. 44, the Mysterious Stranger': 3.88,
    'Kurzgeschichten': 3.75,
    'The Annotated Alice': 4.31,
    'The Escaped Cock': 3.64,
    'Hard Times for These Times': 3.54,
    'Journey to the Centre of the Earth': 3.87,
    'The Castle': 3.99,
    "Enid Blyton's Five Have a Wonderful Time": 4.07,
    'The Quaint and Curious Quest of Johnny Longfoot': 4.14,
    "The Author's Annual": 4.00,
    'Guy Mannering, Or, The Astrologer': 3.58,
    'Her Letter, His Answer & Her Last Letter': 3.75,
    '1601 Conversation as it was by the Social Fireside in the Time of the Tudors': 3.48
}

#mapping the ratings onto the DataFrame using book_name as the pointer
for book_name, rating in ratings_map.items():
    final_df.loc[final_df['book_name'] == book_name, 'rating'] = rating

#dentify and drop serial non-books / journals / indexes
names_to_drop = [
    "Wonderpedia of NeoPopRealism Journal, Today's Featured Articles, 2010-2013",
    "Essay and General Literature Index",
    "The History of Brooklyn's Three Major Performing Arts Institutions",
    "International Directory of Company Histories",
    "Featured Reviews in Mathematical Reviews 1997-1999",
    "Annual Report of the Department of Agriculture, for the Province of Ontario"
]

final_df = final_df[~final_df['book_name'].isin(names_to_drop)] 


## 15. Save the clean catalogue

In [96]:
final_df.info()

<class 'pandas.DataFrame'>
Index: 1083 entries, 675 to 825
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   book_name    1083 non-null   str           
 1   book_url     1083 non-null   str           
 2   source_list  1083 non-null   str           
 3   author       1083 non-null   str           
 4   rating       1083 non-null   float64       
 5   synopsis     1083 non-null   str           
 6   pages        1083 non-null   float64       
 7   pub_date     1083 non-null   datetime64[us]
 8   genres       1083 non-null   str           
 9   img_url      1083 non-null   str           
 10  data_source  1083 non-null   str           
dtypes: datetime64[us](1), float64(2), str(8)
memory usage: 1.2 MB


In [97]:
out_csv(
    df                  = final_df,
    yaml_path           = yaml_path,
    output_section_yaml = 'clean_data',
    file_name           = 'clean'
)

print(f'Saved {len(final_df)} records.')

File saved to: ../data/clean/booksDB.csv
Saved 1083 records.


## 16. Observations & trade-offs

🔍 **Result:** one clean catalogue written to `data/clean/clean.csv` — two independent sources,
three enrichment passes, deduplicated on both title and synopsis.

✅ **What went well**
- Null-filling before dropping preserved several hundred books that a naive `dropna()` would
  have destroyed.
- The Hardcover `ratings_count == 0` gate caught a silent data-poisoning bug: an API returning
  `0` for "no data" instead of `null`.
- Deduplication keeps the *most complete* copy of each book, not an arbitrary one.

⚠️ **Known limitations — carried forward to the README**
1. **Manual ratings (section 14).** ~125 ratings are hand-sourced, unflagged and
   non-reproducible. This is the weakest link in the catalogue.
2. **Source flags are dropped**  the final file cannot attribute a value to the source that supplied it.
3. **Index-addressed repairs** (sections 10, 12, 14) bind this notebook to one specific run of
   the pipeline. Re-scraping invalidates them.
4. **Genre text is still raw and noisy** — 676 unique genre strings. Cleaning happens in
   notebook 4.

➡️ **Next:** `4_EDA_clustering.ipynb`.